<a href="https://colab.research.google.com/github/malihasaeed/langraphai/blob/main/multiagent_competitive_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install -U langgraph langchain langchain-openai pydantic duckduckgo-search wikipedia tenacity -q

In [20]:
import os, time, json
import operator
from typing import TypedDict, List, Dict, Any, Optional, Literal, Annotated

from google.colab import userdata
from pydantic import BaseModel, Field, ValidationError

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

In [21]:
api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY missing from Colab Secrets (🔑). Add it and re-run.")
os.environ["OPENAI_API_KEY"] = api_key
print("✅ OPENAI_API_KEY loaded from Colab Secrets")

✅ OPENAI_API_KEY loaded from Colab Secrets


In [22]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)

In [23]:
class WebSearchInput(BaseModel):
    query: str = Field(..., min_length=3)
    max_results: int = Field(default=5, ge=1, le=10)

class SearchResult(BaseModel):
    title: str
    url: str
    snippet: str

class WebSearchOutput(BaseModel):
    query: str
    results: List[SearchResult]

In [24]:
from duckduckgo_search import DDGS

class ToolError(Exception):
    pass

@retry(
    reraise=True,
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=8),
    retry=retry_if_exception_type(ToolError),
)
def ddg_search(tool_input: WebSearchInput) -> WebSearchOutput:
    try:
        with DDGS() as ddgs:
            raw = list(ddgs.text(tool_input.query, max_results=tool_input.max_results))
        results = []
        for r in raw:
            title = (r.get("title") or "").strip()
            url = (r.get("href") or r.get("url") or "").strip()
            snippet = (r.get("body") or r.get("snippet") or "").strip()
            if title and url:
                results.append(SearchResult(title=title, url=url, snippet=snippet))
        return WebSearchOutput(query=tool_input.query, results=results)
    except Exception as e:
        raise ToolError(str(e))

In [25]:
import wikipedia

class WikiInput(BaseModel):
    topic: str = Field(..., min_length=3)
    sentences: int = Field(default=3, ge=1, le=6)

class WikiOutput(BaseModel):
    topic: str
    summary: str

@retry(
    reraise=True,
    stop=stop_after_attempt(2),
    wait=wait_exponential(multiplier=1, min=1, max=6),
    retry=retry_if_exception_type(ToolError),
)
def wiki_summary(tool_input: WikiInput) -> WikiOutput:
    try:
        wikipedia.set_lang("en")
        summary = wikipedia.summary(tool_input.topic, sentences=tool_input.sentences)
        return WikiOutput(topic=tool_input.topic, summary=summary)
    except Exception as e:
        raise ToolError(str(e))

In [26]:
def run_tool(tool_name: str, fn, tool_input: BaseModel) -> Dict[str, Any]:
    start = time.time()
    try:
        out = fn(tool_input)
        return {
            "ok": True,
            "tool": tool_name,
            "input": tool_input.model_dump(),
            "latency_s": round(time.time() - start, 3),
            "output": out.model_dump()
        }
    except Exception as e:
        return {
            "ok": False,
            "tool": tool_name,
            "input": tool_input.model_dump(),
            "latency_s": round(time.time() - start, 3),
            "error": str(e)
        }

In [27]:
class MultiAgentState(TypedDict):
    request: str

    messages: Annotated[List[str], operator.add]
    tasks: Annotated[List[str], operator.add]
    evidence: Annotated[List[Dict[str, Any]], operator.add]

    analysis: str
    report: str

    confidence: float
    missing_info: List[str]

    step_count: int
    max_steps: int
    last_error: str

    tool_logs: Annotated[List[Dict[str, Any]], operator.add]
    trace: Annotated[List[str], operator.add]
    next_agent: str
    stop: bool

In [28]:
class SupervisorDecision(BaseModel):
    next_agent: Literal["research", "analyst", "report", "clarify", "end"]
    rationale: str = Field(..., min_length=10)
    tasks_to_add: List[str] = Field(default_factory=list)
    stop: bool = False
    confidence: float = Field(default=0.5, ge=0.0, le=1.0)
    missing_info: List[str] = Field(default_factory=list)

supervisor_llm = llm.with_structured_output(SupervisorDecision)

In [29]:
class SupervisorDecision(BaseModel):
    next_agent: Literal["research", "analyst", "report", "clarify", "end"]
    rationale: str = Field(..., min_length=10)
    tasks_to_add: List[str] = Field(default_factory=list)
    stop: bool = False
    confidence: float = Field(default=0.5, ge=0.0, le=1.0)
    missing_info: List[str] = Field(default_factory=list)

supervisor_llm = llm.with_structured_output(SupervisorDecision)
def supervisor_node(state: MultiAgentState):
    if state["step_count"] >= state["max_steps"]:
        return {
            "trace": [f"[supervisor] max_steps reached ({state['max_steps']})"],
            "last_error": "Max steps reached; stopping.",
            "confidence": min(state.get("confidence", 0.0), 0.4),
            "_stop": True,
            "_next_agent": "end"
        }

    prompt = f"""
You are the Supervisor of a multi-agent competitive intelligence system.
Decide the next agent to run based on the shared state.

Request:
{state["request"]}

Tasks (recent):
{state["tasks"][-10:]}

Evidence count: {len(state["evidence"])}
Has analysis: {"yes" if state.get("analysis") else "no"}
Has report: {"yes" if state.get("report") else "no"}

Rules:
- If no evidence yet, choose research.
- If evidence exists but no analysis, choose analyst.
- If analysis exists and report is missing, choose report.
- If request is ambiguous, choose clarify and list missing_info.
- Add tasks_to_add if research needs decomposition.

Return a structured decision.
"""
    decision = supervisor_llm.invoke(prompt)

    return {
        "step_count": state["step_count"] + 1,
        "confidence": decision.confidence,
        "missing_info": decision.missing_info,
        "tasks": decision.tasks_to_add,
        "messages": [f"[Supervisor] {decision.rationale}"],
        "trace": [f"[supervisor] next_agent={decision.next_agent} stop={decision.stop}"],
        "next_agent": decision.next_agent,
        "stop": decision.stop
    }

In [30]:
def research_node(state: MultiAgentState):
    tasks = state["tasks"][-3:] if state["tasks"] else [state["request"]]

    evidence_updates = []
    tool_logs = []
    msg_updates = []

    for t in tasks:
        log = run_tool("ddg_search", ddg_search, WebSearchInput(query=t, max_results=5))
        tool_logs.append(log)

        results = []
        if log["ok"]:
            results = log["output"]["results"]

        if len(results) < 2:
            fb = run_tool("wiki_summary", wiki_summary, WikiInput(topic=t, sentences=3))
            tool_logs.append(fb)
            if fb["ok"]:
                evidence_updates.append({
                    "source": "wikipedia",
                    "query": t,
                    "title": fb["output"]["topic"],
                    "url": "https://en.wikipedia.org/",
                    "snippet": fb["output"]["summary"]
                })
                msg_updates.append(f"[Research] Fallback wiki summary used for: {t}")
            else:
                msg_updates.append(f"[Research] Search failed for: {t}")
            continue

        for r in results:
            evidence_updates.append({
                "source": "web",
                "query": t,
                "title": r["title"],
                "url": r["url"],
                "snippet": r["snippet"]
            })

        msg_updates.append(f"[Research] Collected {len(results)} results for: {t}")

    return {
        "evidence": evidence_updates,
        "tool_logs": tool_logs,
        "messages": msg_updates,
        "trace": [f"[research] evidence_added={len(evidence_updates)}"]
    }

In [31]:
class AnalystOutput(BaseModel):
    analysis: str = Field(..., min_length=50)
    confidence: float = Field(..., ge=0.0, le=1.0)
    additional_tasks: List[str] = Field(default_factory=list)
    missing_info: List[str] = Field(default_factory=list)

analyst_llm = llm.with_structured_output(AnalystOutput)

def analyst_node(state: MultiAgentState):
    ev = state["evidence"][-30:]
    evidence_text = "\n\n".join(
        [f"- ({e['source']}) {e['title']} | {e['url']}\n  {e['snippet']}" for e in ev]
    )

    prompt = f"""
You are the Analyst Agent.
Convert evidence into high-signal competitive intelligence.

Request:
{state["request"]}

Evidence:
{evidence_text}

Requirements:
- Provide structured analysis (competitors, trends, signals, risks)
- Be explicit about uncertainty
- Provide confidence 0–1
- Suggest additional tasks if evidence is weak
"""
    out = analyst_llm.invoke(prompt)

    return {
        "analysis": out.analysis,
        "confidence": out.confidence,
        "tasks": out.additional_tasks,
        "missing_info": out.missing_info,
        "messages": [f"[Analyst] Analysis ready (confidence={out.confidence})"],
        "trace": ["[analyst] analysis_ready"]
    }

In [32]:
def report_node(state: MultiAgentState):
    ev = state["evidence"][-20:]
    citations = list({e["url"] for e in ev if e.get("url")})[:12]

    prompt = f"""
You are the Report Agent. Write an executive-grade competitive intelligence report.

Request:
{state["request"]}

Analyst output:
{state.get("analysis","")}

Citations (URLs to include):
{citations}

Format:
1) Executive Summary (5-7 bullets)
2) Key Competitors (bullets + differentiators)
3) Market/Trend Signals (bullets)
4) Risks & Unknowns (bullets)
5) Recommendations (ranked)
6) Citations (list)

End with overall confidence (0–1).
"""
    report = llm.invoke(prompt).content.strip()

    return {
        "report": report,
        "messages": ["[Report] Report drafted with citations"],
        "trace": ["[report] report_ready"]
    }

In [33]:
def clarify_node(state: MultiAgentState):
    missing = state.get("missing_info") or ["Please clarify scope/region/target segment."]
    print("\n" + "="*90)
    print("CLARIFICATION NEEDED:")
    for q in missing:
        print("-", q)
    print("="*90)
    user_answer = input("Provide clarification: ").strip()

    new_request = state["request"] + "\n\n[User clarification]\n" + user_answer
    return {
        "request": new_request,
        "messages": ["[Clarify] User provided clarification"],
        "trace": ["[clarify] updated_request"]
    }

In [34]:
def route_from_supervisor(state: MultiAgentState):
    if state["stop"]:
        return END

    if state["next_agent"] == "research":
        return "research"
    if state["next_agent"] == "analyst":
        return "analyst"
    if state["next_agent"] == "report":
        return "report"
    if state["next_agent"] == "clarify":
        return "clarify"

    return END

In [35]:
#building the graph n compiling
checkpointer = MemorySaver()

g = StateGraph(MultiAgentState)
g.add_node("supervisor", supervisor_node)
g.add_node("research", research_node)
g.add_node("analyst", analyst_node)
g.add_node("report", report_node)
g.add_node("clarify", clarify_node)

g.set_entry_point("supervisor")
g.add_conditional_edges("supervisor", route_from_supervisor)

g.add_edge("research", "supervisor")
g.add_edge("analyst", "supervisor")
g.add_edge("clarify", "supervisor")
g.add_edge("report", END)

app = g.compile(checkpointer=checkpointer)
print("✅ Multi-agent graph compiled")

✅ Multi-agent graph compiled


In [36]:
initial_state = {
    "request": "Analyze the current competitive landscape for AI-powered customer service solutions, focusing on key players, their market share, and recent product innovations",
    "messages": [],
    "tasks": ["..."],
    "evidence": [],
    "analysis": "",
    "report": "",
    "confidence": 0.0,
    "missing_info": [],
    "step_count": 0,
    "max_steps": 6,
    "last_error": "",
    "tool_logs": [],
    "trace": [],
    "_next_agent": "research",
    "_stop": False
}


result = app.invoke(initial_state, config={"configurable": {"thread_id": "ci-demo-1"}})

print("\n" + "="*90)
print("FINAL REPORT (preview):\n")
print(result.get("report","")[:2000], "..." if len(result.get("report",""))>2000 else "")
print("\n" + "="*90)

print("\nTRACE:")
for t in result["trace"]:
    print("-", t)

print("\nTOOL LOGS (last 5):")
for tl in result["tool_logs"][-5:]:
    print(tl["tool"], "ok=" + str(tl["ok"]), "latency_s=" + str(tl["latency_s"]))

/tmp/ipython-input-1439912714.py:14: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use time


FINAL REPORT (preview):

### Executive Summary
- The AI-powered customer service solutions market is rapidly expanding, projected to reach **$8 billion by 2025** with a **40% CAGR** from 2020 to 2025.
- Key players include **Haptik**, **CoRover.ai**, **Niki.ai**, **OpenAI**, **Google DeepMind**, **Krutrim**, and **Sarvam**, each leveraging unique technologies.
- Recent innovations emphasize **generative AI**, **reinforcement learning**, and enhanced **NLP capabilities** to improve customer interactions.
- There is a growing trend towards **personalization** in customer service, driven by AI advancements and user demand.
- Market saturation and regulatory challenges pose risks for existing and new entrants in the AI customer service space.
- Support from government initiatives is fostering innovation and adoption of AI technologies in India.
- The absence of specific market share data limits comprehensive competitive analysis.

### Key Competitors
- **Haptik**
  - Differentiator: Stron

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Creating `README.md` file

The following cell will write the content of the README into a file named `README.md` in your Colab environment's file system. You can then download it or view it in the files section.

In [37]:
readme_content = """
# Multi-Agent Competitive Intelligence System

This project implements a multi-agent system designed to perform competitive intelligence and generate reports based on a user's request. It utilizes `langgraph` for agent orchestration and `langchain` for LLM integrations, along with external tools for web search and information retrieval.

## Table of Contents
1. Overview
2. Agents and Their Roles
3. Setup Instructions
4. How to Use
5. Dependencies

## Overview

The system consists of several specialized AI agents coordinated by a central Supervisor. The goal is to process a user's request for competitive intelligence, gather relevant information, analyze it, and produce a structured report.

## Agents and Their Roles

There are **5 distinct agents** (or nodes) in this system:

### 1. Supervisor Agent (`supervisor_node`)
*   **Role**: Orchestrates the workflow, deciding which agent runs next based on the current state of the task.
*   **Functionality**: Checks for progress, directs research, analysis, or report generation, and can request clarification if the input is ambiguous.

### 2. Research Agent (`research_node`)
*   **Role**: Gathers raw information from external sources.
*   **Functionality**: Uses DuckDuckGo Search (`ddg_search`) for web queries and falls back to Wikipedia (`wiki_summary`) if web search results are insufficient. Collects and structures evidence.

### 3. Analyst Agent (`analyst_node`)
*   **Role**: Transforms raw evidence into structured competitive analysis.
*   **Functionality**: Synthesizes collected evidence, identifies competitors, market trends, risks, and opportunities. Provides a confidence score and can suggest additional research or flag missing information.

### 4. Report Agent (`report_node`)
*   **Role**: The document generator, responsible for compiling the intelligence into a final, polished report.
*   **Functionality**: Drafts a comprehensive report, integrating the analyst's findings and citing all sources. The report is structured into sections like Executive Summary, Key Competitors, Market/Trend Signals, Risks & Unknowns, Recommendations, and Citations.

### 5. Clarify Agent (`clarify_node`)
*   **Role**: Interacts with the user to get clarification on ambiguous or incomplete requests.
*   **Functionality**: Prompts the user for additional details and incorporates the clarification into the original request for further processing.

## Setup Instructions

1.  **OpenAI API Key**: Ensure you have an `OPENAI_API_KEY` set up in your Google Colab Secrets. Click the '🔑' icon on the left panel, then 'Add a new secret'. Name it `OPENAI_API_KEY` and paste your key.

2.  **Install Dependencies**: The project requires several Python libraries. Run the following command in a code cell:
    ```python
    !pip install -U langgraph langchain langchain-openai pydantic duckduckgo-search wikipedia tenacity -q
    ```

3.  **Import Libraries and Set Up LLM**: Import necessary modules and initialize your LLM (e.g., `ChatOpenAI`).

    ```python
    import os, time, json
    import operator
    from typing import TypedDict, List, Dict, Any, Optional, Literal, Annotated

    from google.colab import userdata
    from pydantic import BaseModel, Field, ValidationError

    from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

    from langchain_openai import ChatOpenAI
    from langgraph.graph import StateGraph, END
    from langgraph.checkpoint.memory import MemorySaver

    api_key = userdata.get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY missing from Colab Secrets (🔑). Add it and re-run.")
    os.environ["OPENAI_API_KEY"] = api_key
    print("✅ OPENAI_API_KEY loaded from Colab Secrets")

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
    ```

## How to Use

1.  **Define Agent Functions and State Schema**: Ensure all agent node functions (`supervisor_node`, `research_node`, `analyst_node`, `report_node`, `clarify_node`) and the `MultiAgentState` schema are defined and executed.

2.  **Compile the Graph**: The `StateGraph` defines the flow. This will be compiled into an `app` object.
    ```python
    # ... (code for defining nodes, edges, etc. as seen in the notebook)
    workflow = StateGraph(MultiAgentState)
    workflow.add_node("supervisor", supervisor_node)
    workflow.add_node("research", research_node)
    # ... add other nodes and define edges
    checkpointer = MemorySaver()
    app = workflow.compile(checkpointer=checkpointer)
    print("✅ Multi-agent graph compiled")
    ```

3.  **Prepare Initial State**: Create an `initial_state` dictionary with your request.
    ```python
    initial_state = {
        "request": "Analyze the current competitive landscape for AI-powered customer service solutions, focusing on key players, their market share, and recent product innovations",
        "messages": [],
        "tasks": [],
        "evidence": [],
        "analysis": "",
        "report": "",
        "confidence": 0.0,
        "missing_info": [],
        "step_count": 0,
        "max_steps": 6, # Maximum steps to prevent infinite loops
        "last_error": "",
        "tool_logs": [],
        "trace": [],
        "next_agent": "supervisor",
        "stop": False
    }
    ```

4.  **Invoke the App**: Run the compiled graph with your `initial_state`.
    ```python
    result = app.invoke(initial_state, config={"configurable": {"thread_id": "ci-demo-1"}})
    ```

5.  **Review Results**: Inspect the `result` dictionary for the generated `report`, `trace` of agent actions, and `tool_logs`.
    ```python
    print("\n" + "="*90)
    print("FINAL REPORT (preview):\n")
    print(result.get("report","")[:2000], "..." if len(result.get("report",""))>2000 else "")
    print("\n" + "="*90)

    print("\nTRACE:")
    for t in result["trace"]:
        print("-", t)

    print("\nTOOL LOGS (last 5):")
    for tl in result["tool_logs"][-5:]:
        print(tl["tool"], "ok=" + str(tl["ok"]), "latency_s=" + str(tl["latency_s"]))
    ```

## Dependencies

*   `langgraph`
*   `langchain`
*   `langchain-openai`
*   `pydantic`
*   `duckduckgo-search`
*   `wikipedia`
*   `tenacity`
"""

with open('README.md', 'w') as f:
    f.write(readme_content)

print("✅ `README.md` created successfully in the file system!")

✅ `README.md` created successfully in the file system!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

# Multi-Agent Competitive Intelligence System

This project implements a multi-agent system designed to perform competitive intelligence and generate reports based on a user's request. It utilizes `langgraph` for agent orchestration and `langchain` for LLM integrations, along with external tools for web search and information retrieval.

## Table of Contents
1. [Overview](#overview)
2. [Agents and Their Roles](#agents-and-their-roles)
3. [Setup Instructions](#setup-instructions)
4. [How to Use](#how-to-use)
5. [Dependencies](#dependencies)

## Overview

The system consists of several specialized AI agents coordinated by a central Supervisor. The goal is to process a user's request for competitive intelligence, gather relevant information, analyze it, and produce a structured report.

## Agents and Their Roles

There are **5 distinct agents** (or nodes) in this system:

### 1. Supervisor Agent (`supervisor_node`)
*   **Role**: Orchestrates the workflow, deciding which agent runs next based on the current state of the task.
*   **Functionality**: Checks for progress, directs research, analysis, or report generation, and can request clarification if the input is ambiguous.

### 2. Research Agent (`research_node`)
*   **Role**: Gathers raw information from external sources.
*   **Functionality**: Uses DuckDuckGo Search (`ddg_search`) for web queries and falls back to Wikipedia (`wiki_summary`) if web search results are insufficient. Collects and structures evidence.

### 3. Analyst Agent (`analyst_node`)
*   **Role**: Transforms raw evidence into structured competitive analysis.
*   **Functionality**: Synthesizes collected evidence, identifies competitors, market trends, risks, and opportunities. Provides a confidence score and can suggest additional research or flag missing information.

### 4. Report Agent (`report_node`)
*   **Role**: Compiles the analysis into a polished, executive-grade report.
*   **Functionality**: Drafts a comprehensive report, integrating the analyst's findings and citing all sources. The report is structured into sections like Executive Summary, Key Competitors, Market/Trend Signals, Risks & Unknowns, Recommendations, and Citations.

### 5. Clarify Agent (`clarify_node`)
*   **Role**: Interacts with the user to get clarification on ambiguous or incomplete requests.
*   **Functionality**: Prompts the user for additional details and incorporates the clarification into the original request for further processing.

## Setup Instructions

1.  **OpenAI API Key**: Ensure you have an `OPENAI_API_KEY` set up in your Google Colab Secrets. Click the '🔑' icon on the left panel, then 'Add a new secret'. Name it `OPENAI_API_KEY` and paste your key.

2.  **Install Dependencies**: The project requires several Python libraries. Run the following command in a code cell:
    ```python
    !pip install -U langgraph langchain langchain-openai pydantic duckduckgo-search wikipedia tenacity -q
    ```

3.  **Import Libraries and Set Up LLM**: Import necessary modules and initialize your LLM (e.g., `ChatOpenAI`).

    ```python
    import os, time, json
    import operator
    from typing import TypedDict, List, Dict, Any, Optional, Literal, Annotated

    from google.colab import userdata
    from pydantic import BaseModel, Field, ValidationError

    from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

    from langchain_openai import ChatOpenAI
    from langgraph.graph import StateGraph, END
    from langgraph.checkpoint.memory import MemorySaver

    api_key = userdata.get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY missing from Colab Secrets (🔑). Add it and re-run.")
    os.environ["OPENAI_API_KEY"] = api_key
    print("✅ OPENAI_API_KEY loaded from Colab Secrets")

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
    ```

## How to Use

1.  **Define Agent Functions and State Schema**: Ensure all agent node functions (`supervisor_node`, `research_node`, `analyst_node`, `report_node`, `clarify_node`) and the `MultiAgentState` schema are defined and executed.

2.  **Compile the Graph**: The `StateGraph` defines the flow. This will be compiled into an `app` object.
    ```python
    # ... (code for defining nodes, edges, etc. as seen in the notebook)
    workflow = StateGraph(MultiAgentState)
    workflow.add_node("supervisor", supervisor_node)
    workflow.add_node("research", research_node)
    # ... add other nodes and define edges
    checkpointer = MemorySaver()
    app = workflow.compile(checkpointer=checkpointer)
    print("✅ Multi-agent graph compiled")
    ```

3.  **Prepare Initial State**: Create an `initial_state` dictionary with your request.
    ```python
    initial_state = {
        "request": "Analyze the current competitive landscape for AI-powered customer service solutions, focusing on key players, their market share, and recent product innovations",
        "messages": [],
        "tasks": [],
        "evidence": [],
        "analysis": "",
        "report": "",
        "confidence": 0.0,
        "missing_info": [],
        "step_count": 0,
        "max_steps": 6, # Maximum steps to prevent infinite loops
        "last_error": "",
        "tool_logs": [],
        "trace": [],
        "next_agent": "supervisor",
        "stop": False
    }
    ```

4.  **Invoke the App**: Run the compiled graph with your `initial_state`.
    ```python
    result = app.invoke(initial_state, config={"configurable": {"thread_id": "ci-demo-1"}})
    ```

5.  **Review Results**: Inspect the `result` dictionary for the generated `report`, `trace` of agent actions, and `tool_logs`.
    ```python
    print("\n" + "="*90)
    print("FINAL REPORT (preview):\n")
    print(result.get("report","")[:2000], "..." if len(result.get("report",""))>2000 else "")
    print("\n" + "="*90)

    print("\nTRACE:")
    for t in result["trace"]:
        print("-", t)

    print("\nTOOL LOGS (last 5):")
    for tl in result["tool_logs"][-5:]:
        print(tl["tool"], "ok=" + str(tl["ok"]), "latency_s=" + str(tl["latency_s"]))
    ```

## Dependencies

*   `langgraph`
*   `langchain`
*   `langchain-openai`
*   `pydantic`
*   `duckduckgo-search`
*   `wikipedia`
*   `tenacity`

## Understanding the Multi-Agent System

This system is designed to perform competitive intelligence by orchestrating several specialized AI agents. There are **5 distinct agents** (or nodes) that work together, managed by a central Supervisor.

Here's a breakdown of each agent's role and function:

### 1. Supervisor Agent (Node: `supervisor_node`)

*   **Role**: The brain of the operation, responsible for managing the workflow and deciding the next steps.
*   **What it does**:
    *   **Orchestrates**: Directs traffic between other agents based on the current state of the task (e.g., if more research is needed, if analysis is ready, if a report should be drafted).
    *   **Monitors Progress**: Keeps track of the `step_count` to prevent infinite loops and ensures the system is moving towards completion.
    *   **Evaluates State**: Assesses the presence of `evidence`, `analysis`, and `report` to determine the most appropriate next action.
    *   **Clarification**: If the initial `request` is unclear or vital `missing_info` is identified by other agents, it can route the task to the `clarify_node`.
    *   **Decision Making**: Uses an LLM to make informed decisions about which agent to activate next, providing a `rationale` and potentially adding new `tasks_to_add`.

### 2. Research Agent (Node: `research_node`)

*   **Role**: The information gatherer, responsible for collecting raw data from external sources.
*   **What it does**:
    *   **Web Search**: Utilizes `ddg_search` (DuckDuckGo Search) to find relevant information based on current `tasks` or the initial `request`.
    *   **Fallback Search**: If web search results are insufficient, it falls back to `wiki_summary` (Wikipedia) to get a brief overview of the topic.
    *   **Evidence Collection**: Processes the search results and Wikipedia summaries, structuring them as `evidence` (title, URL, snippet, source) to be used by other agents.
    *   **Tool Logging**: Records the execution and outcomes of the search tools in `tool_logs`.

### 3. Analyst Agent (Node: `analyst_node`)

*   **Role**: The intelligence processor, tasked with transforming raw evidence into structured analysis.
*   **What it does**:
    *   **Synthesizes Evidence**: Takes the `evidence` gathered by the `research_node` as input.
    *   **Generates Analysis**: Uses an LLM to perform in-depth analysis, identifying key competitors, market trends, signals, and potential risks.
    *   **Assesses Confidence**: Provides a `confidence` score (0-1) for its analysis, indicating its certainty.
    *   **Identifies Gaps**: Can suggest `additional_tasks` for the `research_node` or flag `missing_info` if the evidence is incomplete or ambiguous, prompting further investigation or clarification.

### 4. Report Agent (Node: `report_node`)

*   **Role**: The document generator, responsible for compiling the intelligence into a final, polished report.
*   **What it does**:
    *   **Drafts Report**: Uses an LLM to create an executive-grade competitive intelligence `report`.
    *   **Integrates Analysis**: Incorporates the structured `analysis` provided by the `analyst_node`.
    *   **Cites Sources**: Includes relevant URLs from the collected `evidence` as `citations` to support the report's claims.
    *   **Formats Output**: Structures the report into distinct sections such as Executive Summary, Key Competitors, Market/Trend Signals, Risks & Unknowns, Recommendations, and Citations.

### 5. Clarify Agent (Node: `clarify_node`)

*   **Role**: The user interaction specialist, designed to seek user input when the task is unclear.
*   **What it does**:
    *   **Prompts User**: Presents `missing_info` questions (identified by other agents, often the Supervisor or Analyst) to the user via an input prompt.
    *   **Updates Request**: Incorporates the user's `clarification` into the original `request`, making the task more specific for subsequent processing by other agents.